In [4]:
# import basic libraries
import os
import numpy as np
import pandas as pd
import re
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
import pickle
import plotly.graph_objects as go
from collections import Counter
import plotly.io as pio
import pyperclip
from sklearn.cluster import KMeans

In [2]:
# import the Czech BERT model
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("fav-kky/FERNET-C5")
model = AutoModel.from_pretrained("fav-kky/FERNET-C5")

/home/jupyter-vojta/notebooks/scriptum/scriptum_venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7074.58it/s]
[transformers] BertModel LOAD REPORT from: fav-kky/FERNET-C5
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	c

In [5]:
# load metadata table for individual files from scriptum
files_df = pd.read_json("../data/files_Ns_df.json")
#files_df.drop_duplicates(subset="filename", keep="first", inplace=True)

In [6]:
files_df.head(5)

,periodical_category,periodical_href,periodical_title,periodical_basic_info,file_url,filename,bezpečí_N,bezpečnost_N
0,samizdat,/cs/periodika/ps-psi,Ψ [Psí],"[, [Praha] , 1985 – 1989 , 1 × ročně , ]",https://files.scriptum.cz/scriptum/%5Bnode%5D/...,psi_1986_2_vina_ocr.txt,0,NaN
1,samizdat,/cs/periodika/ps-psi,Ψ [Psí],"[, [Praha] , 1985 – 1989 , 1 × ročně , ]",https://files.scriptum.cz/scriptum/%5Bnode%5D/...,psi_1987_3_freud-a-psychoanalyza_ocr.txt,1,NaN
2,samizdat,/cs/periodika/ps-psi,Ψ [Psí],"[, [Praha] , 1985 – 1989 , 1 × ročně , ]",https://files.scriptum.cz/scriptum/%5Bnode%5D/...,psi_1988_4_sluzba-zivotu_rok-blahoslavene-anez...,0,NaN
3,samizdat,/cs/periodika/ps-psi,Ψ [Psí],"[, [Praha] , 1985 – 1989 , 1 × ročně , ]",https://files.scriptum.cz/scriptum/%5Bnode%5D/...,psi_1989_5_vira-v-modernim-svete_ocr.txt,0,NaN
4,samizdat,/cs/periodika/abecedni-casopis,Abecední časopis,"[, Praha , 1985 - 1986 , nepravidelně , ]",https://files.scriptum.cz/scriptum/abecedni-ca...,abecedni-casopis_1986_2_bonifac_tit-str-obsah_...,0,NaN


In [7]:
# navigate to the path with individual text files
path = "../data/rawtexts/"
filenames = os.listdir(path)
len(filenames)

11615

The following code does this:
* For each document, wload a text file with sentences containing the target term (e.g. "bezpečí"). These files contain a sentence per line.
* Go throuh each sentence in the list of sentences and for each 
    * Apply BERT tokenizer on the sentence (see `sentence_data["sentence_tokens"]`).
    * Assign sentence-level BERT embedding to it, averaging the last hidden state layer of the model (see `sentence_data["sentence_embeddings"]`).
    * Assing token-level BERT embedding to each token (see `sentence_data["sentence_token_embeddings"]`). 
    * Select the embedding of the  target term and save it separately (see `sentence_data["target_embedding"]`
    * Assign additional metadata to each sentence based on the source document.

In [8]:
sent = "Tohle je česká věta obsahující můj termín bezpečí."
inputs = tokenizer(sent, return_tensors='pt', padding=True, truncation=True, max_length=512)

In [9]:
tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

['[CLS]',
 'To',
 '##hle',
 'je',
 'česká',
 'věta',
 'obsahující',
 'můj',
 'termín',
 'bezpečí',
 '.',
 '[SEP]']

In [10]:
with torch.no_grad():
    outputs = model(**inputs)
    outputs.last_hidden_state.mean(dim=1)

In [11]:
sentence_token_embeddings = outputs.last_hidden_state[0].numpy()

In [12]:
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

target = "bezpečí"

for idx, token in enumerate(tokens):
    if re.search(target, token, re.IGNORECASE):
        target_embedding = sentence_token_embeddings[idx]
target_embedding

array([-1.09763801e+00, -1.43324688e-01, -1.73250824e-01, -6.01456583e-01,
        6.26601279e-02, -6.44099593e-01, -2.82022625e-01, -2.01308799e+00,
        5.62766902e-02, -3.89248431e-01,  4.22246270e-02,  4.35212582e-01,
       -8.53796363e-01, -1.00658286e+00, -3.46252546e-02, -4.92114812e-01,
        1.24880302e+00,  3.60793740e-01,  3.39815736e-01,  2.57925332e-01,
       -1.19261289e+00,  8.17864597e-01, -9.84924510e-02,  5.92112601e-01,
       -6.02965176e-01, -1.00598283e-01, -8.42973948e-01, -6.51675820e-01,
       -2.86317974e-01,  2.86189944e-01, -8.57163608e-01, -7.37018347e-01,
        3.70829284e-01,  4.13715780e-01, -8.36958528e-01,  1.43151391e+00,
       -7.69851625e-01,  1.19547939e+00,  6.76205516e-01, -7.07372367e-01,
       -6.20760024e-01, -2.02443123e+00,  4.01802063e+00,  6.96048200e-01,
        8.25455666e-01, -1.47094774e+00, -1.27092257e-01,  6.69597924e-01,
       -1.07523274e+00, -9.61499751e-01, -4.28751223e-02, -1.35304499e+00,
        6.12795800e-02, -

In [13]:
sent = "bezpečí"
inputs = tokenizer(sent, return_tensors='pt', padding=True, truncation=True, max_length=512)
with torch.no_grad():
    outputs = model(**inputs)
    outputs.last_hidden_state.mean(dim=1)
isolated_embedding = outputs.last_hidden_state[0].numpy()[2]
isolated_embedding

array([-1.01814854e+00, -1.13664258e+00, -3.87226611e-01, -1.29448819e+00,
       -3.41275573e-01, -1.55991364e+00, -5.32281220e-01, -1.52227998e+00,
       -8.26405063e-02,  2.52543718e-01, -7.99941540e-01,  7.69875646e-01,
       -2.08159900e+00, -1.49639952e+00,  4.94031936e-01, -7.61152327e-01,
        1.38506554e-02,  2.53197491e-01,  2.17934775e+00,  5.82401156e-01,
       -2.83674538e-01, -4.53464001e-01,  9.68107209e-02,  7.05165714e-02,
       -2.30102777e-01, -3.02323233e-02, -5.31813204e-01,  5.09422868e-02,
       -1.28812170e+00,  9.56018746e-01, -5.13344705e-02, -2.22966015e-01,
        1.53554392e+00, -3.10996221e-03, -2.51401091e+00, -3.80260140e-01,
        6.14223421e-01,  8.99643242e-01, -1.02112494e-01,  1.71760306e-01,
        7.69521743e-02, -4.82239008e-01,  4.55606997e-01, -1.52815834e-01,
        3.57810020e-01, -1.62502623e+00, -8.20395723e-03,  4.96669084e-01,
       -8.31236303e-01,  1.26367047e-01, -5.70577621e-01, -2.65403461e+00,
        1.33995986e+00, -

In [14]:
cosine_similarity([target_embedding, isolated_embedding])

array([[1.0000002 , 0.4926011 ],
       [0.4926011 , 0.99999976]], dtype=float32)

In [15]:
# navigate to path with files hosting all sentences containing the term "bezpečí"
input_path = "../data/rawsenteces_bezpeci/"
output_dir = "../data/embeddings_bezpeci/"
try:
    os.mkdir(output_dir)
except:
    pass

def get_embeddings_data(row, target, output_dir):
    try:
        with open(input_path + row["filename"]) as f:
            sentences = f.readlines()
        data = []
        for sent in sentences:
            inputs = tokenizer(sent, return_tensors='pt', padding=True, truncation=True, max_length=512)
            # Convert input_ids to actual tokens
            sentence_data = {
                "filename" : row["filename"],
                "periodical_title" : row["periodical_title"],
                "periodical_category" : row["periodical_category"],
                "year" : row["year"],
            }
            sentence_data["sentence"] = sent
            sentence_data["sentence_tokens"] = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
            sentence_data["token_ids"] = inputs['input_ids'][0].tolist()
            with torch.no_grad():
                outputs = model(**inputs)
                sentence_data["sentence_embeddings"] = outputs.last_hidden_state.mean(dim=1)
                sentence_data["sentence_token_embeddings"] = outputs.last_hidden_state[0].numpy()
            # embedding for a target term
            for idx, token in enumerate(sentence_data["sentence_tokens"]):
                if re.search(target, token, re.IGNORECASE):
                    sentence_data["target_embedding"] = sentence_data["sentence_token_embeddings"][idx]
                    data.append(sentence_data)
        with open(output_dir + row["filename"].replace(".txt", ".pickle"), "wb") as f:
            pickle.dump(data, f)
    except:
        pass

### Analysis of "bezpečí" embeddings

In [16]:
%%time
input_path = "../data/rawsentences_bezpeci/"
output_dir = "../data/embeddings_bezpeci/"
target = r"\bbezpečím?"
files_df.apply(lambda row: get_embeddings_data(row, target, output_dir), axis=1)

CPU times: user 637 ms, sys: 60.1 ms, total: 697 ms
Wall time: 818 ms


0        None
1        None
2        None
3        None
4        None
         ... 
11612    None
11613    None
11614    None
11615    None
11616    None
Length: 11617, dtype: object

In [19]:
data = []
output_dir = "../data/embeddings_bezpeci/"
for filename in os.listdir("../data/embeddings_bezpeci/"):
    data.extend(pickle.load(open(output_dir + filename, "rb")))

passed: studie_1985_100-101_ocr.pickle
passed: narodni-politika_1985_06_ocr.pickle
passed: paternoster_1987_19_ocr.pickle
passed: americke-listy_1987_10-23_28_ocr.pickle
passed: zpravodaj-scss_2000-01_33_12_ocr.pickle
passed: americke-listy_1974_02-22_08_ocr.pickle
passed: hlasy-cechu-a-slovaku-v-australii_1983_17_ocr.pickle
passed: hlas-domova_1955-06_ocr.pickle
passed: ceske-slovo_1970_07_ocr.pickle
passed: pravo-lidu_1988_1_ocr.pickle
passed: hlas-domova_1962_13_ocr.pickle
passed: zpravodaj-scss_2015-16_48_02_ocr.pickle
passed: americke-listy_1974_10-04_40_ocr.pickle
passed: zapad_1983_1_ocr.pickle
passed: hlas-domova_1955_25-26_ocr.pickle
passed: hlas-domova_1970_06_ocr.pickle
passed: sklizen_1955_02_26_ocr.pickle
passed: americke-listy_1989-90_10-13_27_ocr.pickle
passed: zpravodaj-scss_2001-02_34_06_ocr.pickle
passed: ceske-slovo_1981_10_ocr.pickle
passed: lazensky_host_1979_02_ocr.pickle
passed: kalex_1974_ocr.pickle
passed: narodni-politika_1984_11_ocr.pickle
passed: polygon_rep

In [5]:
len(data)

2782

In [6]:
data[3]["filename"]

'narodni-politika_1990_09_ocr.txt'

In [7]:
data[3]["year"]

1990.0

In [8]:
data = [d for d in data if 1948 <= d["year"] <= 1989]

target_vectors = [d["target_embedding"] for d in data]
sentences = [d["sentence"] for d in data]
periodical_categories = [d["periodical_category"] for d in data]
colors = ["green" if x=="exil" else "red" for x in periodical_categories]
hover_text = [d["sentence"].replace("\n", "") + " ({0}, {1}, {2})".format(d["periodical_title"], str(d["year"]).replace(".0", ""), d["periodical_category"])
              for d in data]

In [9]:
Counter(periodical_categories)

Counter({'exil': 1435, 'samizdat': 323})

In [10]:
hover_text[:10]

[' Bude rušit její pocit bezpečí, ohmožsovat její mír, platit za kazisvěta. (Obsah, 1984, samizdat)',
 '“ „Pokud bude možné, že zbraně ze Sovětského svazu a Československa jsou pašovány na západ, dotud není možné hovořili o bezpečí jednotlivých občanů a států na západě. (Národní politika, 1972, exil)',
 ' Javor, jehož úcta k českému slovu je v jeho verších až rafinovaně vybíravá, ve svém spěchu chytat děj, si v ní dovoluje obraty zprofanované v próze špatné, jako když třeba píše, že jeho hrdina Pavel Radvan „myslil spise na vlahé Květiny rty“, žc „psal kouzelně křehké verše“ nebo žc Praha v květnu 1945 „ssála prvé doušky svobody, bezpečí a míru“ a podobně. (České slovo, 1967, exil)',
 ' Negativní stránkou jsou problémy spojené hlavně s kvalitou a bezpečím našeho životního prostředí. (SVU Zprávy, 1989, exil)',
 ' Řekl jsi si: čím víc moci, tím větší bezpečí? Bylo to proto, že jsi hromadil víc a víc moci ve svých rukách, že jsi se opevňoval ve své pozici — někdy nepoctivě? Za starých čas

In [12]:
k = 5
kmeans = KMeans(n_clusters=k, random_state=0).fit(target_vectors)
labels = kmeans.labels_

/opt/homebrew/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [13]:
len(labels)

1758

In [14]:
print(labels[:20])

[2 2 2 0 1 4 3 2 2 3 1 3 4 2 3 4 0 2 3 2]


In [15]:
cluster_colors = {0: "#1f77b4",  # muted blue
                  1: "#ff7f0e",  # safety orange
                  2: "#2ca02c",  # cooked asparagus green
                  3: "#d62728",  # brick red
                  4: "#9467bd"}  # muted purple

In [16]:
colors = [cluster_colors[l] for l in labels]

In [17]:
target_embeddings = np.vstack(target_vectors)
cosine_sim_matrix = pd.DataFrame(cosine_similarity(target_embeddings), index=sentences, columns=sentences)

In [18]:
cosine_sim_matrix.iloc[8].sort_values(ascending=False)

 A tak statečnost a pravda stojí proti bezpečí, klidu, práci, hmotné zajíšíínosti, proti rodině a normální budoucnosti dětí, proti vyřazenosti, veřejným pomluvám a desítkám jiných příkoří, která znají nejlépe ti, kteří se už rozhodli.\n                           1.000000
 A tak statečnost a pravda stojí proti bezpečí, klidu, práci, hmotné zajištěnosti, proti rodině a normální budoucnosti dělí, proti vyřazenosti, veřejným pomluvám a desítkám jiných příkoří, které znají nejlépe ti, kteří se už rozhodli.\n                           0.947904
 Konání dobra a odpor ke zlu chápeme pouze jako prostředek k naší vlastní svobodě, bezpečí a blahobytu a nikoliv jako něco, co má smysl samo o sobě.\n                                                                                                                 0.730190
 který prožil svůj život v bezpečí, klidu a liché důstojnosti v léto politicky zmítané zemi mi prozradil s jistou hrdostí recept na takový život.\n                                     

In [22]:
tsne = TSNE(n_components=3, perplexity=5)
embeddings_tsne = tsne.fit_transform(np.vstack(target_vectors))

In [23]:
xs = embeddings_tsne[:, 0]
ys = embeddings_tsne[:, 1]
zs = embeddings_tsne[:, 2]

In [24]:
title = "Sentences with 'bezpečí' (N={})".format(len(sentences))
fig = go.Figure(data=go.Scatter3d(
    x=xs,
    y=ys,
    z=zs,
    mode='markers',
    marker=dict(
        size=5,
        color=colors,
        opacity=0.3
    ),
    text=hover_text,  # use mapped hover text
    hoverinfo='text',  # ensure only the text field is displayed on hover
))

fig.update_layout(
    title=title,
    scene=dict(
        xaxis=dict(title='', showgrid=False, showline=False, showticklabels=False, zeroline=False, linecolor='rgba(0,0,0,0)'),
        yaxis=dict(title='', showgrid=False, showline=False, showticklabels=False, zeroline=False, linecolor='rgba(0,0,0,0)'),
        zaxis=dict(title='', showgrid=False, showline=False, showticklabels=False, zeroline=False, linecolor='rgba(0,0,0,0)'),
        bgcolor='rgba(255,255,255,0)'
    ),
    paper_bgcolor='rgba(255,255,255,255)',  # set the color of the area around the axes
    plot_bgcolor='rgba(255,255,255,255)',  # set the color of the entire chart
    autosize=False,
    width=1000,
    height=800,
    margin=dict(l=0, r=0, b=0, t=0)
)


In [25]:
html_code = pio.to_html(fig, include_plotlyjs='cdn')
pyperclip.copy(html_code)

In [26]:
colors = ["green" if x=="exil" else "red" for x in periodical_categories]
title = "Sentences with 'bezpečí' (N={})".format(len(sentences))
fig = go.Figure(data=go.Scatter3d(
    x=xs,
    y=ys,
    z=zs,
    mode='markers',
    marker=dict(
        size=5,
        color=colors,
        opacity=0.3
    ),
    text=hover_text,  # use mapped hover text
    hoverinfo='text',  # ensure only the text field is displayed on hover
))

fig.update_layout(
    title=title,
    scene=dict(
        xaxis=dict(title='', showgrid=False, showline=False, showticklabels=False, zeroline=False, linecolor='rgba(0,0,0,0)'),
        yaxis=dict(title='', showgrid=False, showline=False, showticklabels=False, zeroline=False, linecolor='rgba(0,0,0,0)'),
        zaxis=dict(title='', showgrid=False, showline=False, showticklabels=False, zeroline=False, linecolor='rgba(0,0,0,0)'),
        bgcolor='rgba(255,255,255,0)'
    ),
    paper_bgcolor='rgba(255,255,255,255)',  # set the color of the area around the axes
    plot_bgcolor='rgba(255,255,255,255)',  # set the color of the entire chart
    autosize=False,
    width=1000,
    height=800,
    margin=dict(l=0, r=0, b=0, t=0)
)

In [27]:
html_code = pio.to_html(fig, include_plotlyjs='cdn')
pyperclip.copy(html_code)

### Analysis of "bezpečnost" embeddings

In [90]:
%%time
input_path = "../data/rawsentences_bezpecnost/"
output_dir = "../data/embeddings_bezpecnost/"
try:
    os.mkdir(output_dir)
except:
    pass
target = r"bezpečnost[ií]?"
files_df.apply(lambda row: get_embeddings_data(row, target, output_dir), axis=1)

CPU times: user 17min 43s, sys: 6min 58s, total: 24min 42s
Wall time: 22min 49s


0        None
1        None
2        None
3        None
4        None
         ... 
11612    None
11613    None
11614    None
11615    None
11616    None
Length: 11614, dtype: object

In [29]:
data = []
for filename in os.listdir("../data/embeddings_bezpecnost/"):
    data.extend(pickle.load(open("../data/embeddings_bezpecnost/" + filename, "rb")))

# filter by year
data = [d for d in data if 1948 <= d["year"] <= 1989]

target_vectors = [d["target_embedding"] for d in data]
sentences = [d["sentence"] for d in data]
periodical_categories = [d["periodical_category"] for d in data]
colors = ["green" if x=="exil" else "red" for x in periodical_categories]
hover_text = [d["sentence"].replace("\n", "") + " ({0}, {1}, {2})".format(d["periodical_title"], str(d["year"]).replace(".0", ""), d["periodical_category"])
              for d in data]

In [30]:
Counter(periodical_categories)

Counter({'exil': 15198, 'samizdat': 2075})

In [31]:
tsne = TSNE(n_components=3, perplexity=5)
embeddings_tsne = tsne.fit_transform(np.vstack(target_vectors))

In [94]:
xs = embeddings_tsne[:, 0]
ys = embeddings_tsne[:, 1]
zs = embeddings_tsne[:, 2]

In [95]:
title = "Sentences with 'bezpečnost' (N={})".format(len(sentences))
fig = go.Figure(data=go.Scatter3d(
    x=xs,
    y=ys,
    z=zs,
    mode='markers',
    marker=dict(
        size=5,
        color=colors,
        opacity=0.3
    ),
    text=hover_text,  # use mapped hover text
    hoverinfo='text',  # ensure only the text field is displayed on hover
))

fig.update_layout(
    title=title,
    scene=dict(
        xaxis=dict(title='', showgrid=False, showline=False, showticklabels=False, zeroline=False, linecolor='rgba(0,0,0,0)'),
        yaxis=dict(title='', showgrid=False, showline=False, showticklabels=False, zeroline=False, linecolor='rgba(0,0,0,0)'),
        zaxis=dict(title='', showgrid=False, showline=False, showticklabels=False, zeroline=False, linecolor='rgba(0,0,0,0)'),
        bgcolor='rgba(255,255,255,0)'
    ),
    paper_bgcolor='rgba(255,255,255,255)',  # set the color of the area around the axes
    plot_bgcolor='rgba(255,255,255,255)',  # set the color of the entire chart
    autosize=False,
    width=1000,
    height=800,
    margin=dict(l=0, r=0, b=0, t=0)
)


In [96]:
html_code = pio.to_html(fig, include_plotlyjs='cdn')
pyperclip.copy(html_code)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [32]:
data = [d for d in data if d["periodical_category"] == "samizdat"]

target_vectors = [d["target_embedding"] for d in data]
sentences = [d["sentence"] for d in data]
periodical_categories = [d["periodical_category"] for d in data]
colors = ["green" if x=="exil" else "red" for x in periodical_categories]
hover_text = [d["sentence"].replace("\n", "") + " ({0}, {1}, {2})".format(d["periodical_title"], str(d["year"]).replace(".0", ""), d["periodical_category"])
              for d in data]

In [42]:
k = 5
kmeans = KMeans(n_clusters=k, random_state=0).fit(target_vectors)
labels = kmeans.labels_
cluster_colors = {0: "#1f77b4",  # muted blue
                  1: "#ff7f0e",  # safety orange
                  2: "#2ca02c",  # cooked asparagus green
                  3: "#d62728",  # brick red
                  4: "#9467bd"}  # muted purple
colors = [cluster_colors[l] for l in labels]

/opt/homebrew/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:1416: FutureWarning:

The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning



In [43]:
tsne = TSNE(n_components=3, perplexity=3)
embeddings_tsne = tsne.fit_transform(np.vstack(target_vectors))

In [44]:
xs = embeddings_tsne[:, 0]
ys = embeddings_tsne[:, 1]
zs = embeddings_tsne[:, 2]

In [51]:
title = "Sentences with 'bezpečnost' (N={})".format(len(sentences))
fig = go.Figure(data=go.Scatter3d(
    x=xs,
    y=ys,
    z=zs,
    mode='markers',
    marker=dict(
        size=5,
        color=colors,
        opacity=0.3
    ),
    text=hover_text,  # use mapped hover text
    hoverinfo='text',  # ensure only the text field is displayed on hover
))

fig.update_layout(
    title=title,
    scene=dict(
        xaxis=dict(title='', showgrid=False, showline=False, showticklabels=False, zeroline=False, linecolor='rgba(0,0,0,0)'),
        yaxis=dict(title='', showgrid=False, showline=False, showticklabels=False, zeroline=False, linecolor='rgba(0,0,0,0)'),
        zaxis=dict(title='', showgrid=False, showline=False, showticklabels=False, zeroline=False, linecolor='rgba(0,0,0,0)'),
        bgcolor='rgba(255,255,255,0)'
    ),
    paper_bgcolor='rgba(255,255,255,255)',  # set the color of the area around the axes
    plot_bgcolor='rgba(255,255,255,255)',  # set the color of the entire chart
    autosize=False,
    width=1000,
    height=800,
    margin=dict(l=0, r=0, b=0, t=0)
)

In [53]:
html_code = pio.to_html(fig, include_plotlyjs='cdn')
pyperclip.copy(html_code)